In [ ]:
!pip install -U bitsandbytes>=0.46.1

In [ ]:
from transformers import (
    pipeline,
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig
)
from peft import PeftModel
import torch
import pandas as pd

In [ ]:
BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
ADAPTER = "PranayCh/CustSupportQandA2026-09-05_07.38.49"


tokenizer = AutoTokenizer.from_pretrained(ADAPTER)


quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)


base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quantization_config,
    device_map="auto"
)


model = PeftModel.from_pretrained(
    base_model,
    ADAPTER
)

In [ ]:

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)


input_file = "/content/testing50Questions.csv"

df = pd.read_csv(input_file)

question_column = df.columns[0]

results = []


for i, question in enumerate(df[question_column]):

    output = generator(
        [
            {
                "role": "user",
                "content": str(question)
            }
        ],
        max_new_tokens=512,
        return_full_text=False
    )

    generated_answer = output[0]["generated_text"]

    results.append({
        "question": question,
        "generated_answer": generated_answer
    })

    print(f"Processed {i + 1}/{len(df)}")


results_df = pd.DataFrame(results)

output_file = "finetuned_model_outputs.csv"

results_df.to_csv(
    output_file,
    index=False
)

print(f"File: {output_file}")